<a href="https://colab.research.google.com/github/trulydeeprogrmr29/Sentiment_Analysis_Project/blob/main/Sentiment_Analysis_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Ingestion

In [1]:
import pandas as pd

# Fetching  IMDb sentiment dataset
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url)


print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 2. Exploratory Data Analysis

In [2]:
# Check the distribution of sentiments
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [4]:
import re

def clean_text(text):
    # 1. Remove HTML line breaks and tags
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [5]:
df['cleaned_review'] = df['review'].apply(clean_text)

print("ORIGINAL:")
print(df['review'].iloc[1][:150])
print("\nCLEANED:")
print(df['cleaned_review'].iloc[1][:150])

ORIGINAL:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes d

CLEANED:
a wonderful little production the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sens


##Vectorization & Data Splitting

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

# Learn the vocabulary and transform the cleaned reviews into numerical features
X = vectorizer.fit_transform(df['cleaned_review'])
y = df['sentiment']

print(f"Feature Matrix Shape: {X.shape}")
print(f"Type of matrix: {type(X)}")

Feature Matrix Shape: (50000, 5000)
Type of matrix: <class 'scipy.sparse._csr.csr_matrix'>


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data size: {X_train.shape[0]} reviews")
print(f"Testing data size: {X_test.shape[0]} reviews")

Training data size: 40000 reviews
Testing data size: 10000 reviews


##Model Training & Evaluation


In [9]:
from sklearn.linear_model import LogisticRegression

# Initialize the model
# We use class_weight='balanced' as a best practice, though our data is already clean!
model = LogisticRegression(max_iter=1000, class_weight='balanced')

# Train the model on the 80% training data
model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

# Predict sentiments for the unseen test data
y_pred = model.predict(X_test)

#  performance metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))

Overall Accuracy: 89.29%

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.89      0.89      5000
    positive       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



##Custom Testing & Model Export

In [11]:
def predict_sentiment(text):
    # 1. Clean the incoming text
    cleaned = clean_text(text)

    # 2. Transform the text using the existing trained vectorizer
    # Note: Wrap text in a list [cleaned] because the vectorizer expects an iterable
    vector = vectorizer.transform([cleaned])

    # 3. Predict and grab the string label out of the resulting array
    prediction = model.predict(vector)[0]

    return prediction

# Test cases - feel free to change these phrases!
print(f"Test 1: {predict_sentiment('This movie was an absolute masterpiece, acting was top tier!')}")
print(f"Test 2: {predict_sentiment('Complete waste of time. I fell asleep halfway through.')}")
print(f"Test 3: {predict_sentiment('It wasn\'t the worst thing ever, but I wouldn\'t watch it again.')}")

Test 1: positive
Test 2: negative
Test 3: negative


In [13]:
import pickle

# Save
with open("model.pkl", "wb") as model_file:
    pickle.dump(model, model_file)

# Save the TF-IDF vectorizer configuration
with open("vectorizer.pkl", "wb") as vec_file:
    pickle.dump(vectorizer, vec_file)

print("Files saved successfully as 'model.pkl' and 'vectorizer.pkl'!")

Files saved successfully as 'model.pkl' and 'vectorizer.pkl'!


In [16]:
import numpy as np

def smart_router_predict(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])

    # Get probability scores: [Probability of Negative, Probability of Positive]
    probabilities = model.predict_proba(vector)[0]
    max_confidence = np.max(probabilities)
    predicted_class = model.predict(vector)[0]

    print(f"Text: '{text}'")
    print(f"Local Model Prediction: {predicted_class} (Confidence: {max_confidence*100:.2f}%)")

    # SET A ROUTING THRESHOLD (e.g., 80%)
    if max_confidence < 0.80:
        print(" Status: Local confidence too low! Routing to Claude/Gemini API...")
        # This is where your API calling function would run
        # return call_llm_api(text)
    else:
        print("Status: Local prediction accepted.")


# Test with a clear phrase vs a tricky, nuanced phrase
smart_router_predict("This movie was absolutely incredible and stunning!")
smart_router_predict("It wasn't the worst thing ever, but I wouldn't watch it again.")

Text: 'This movie was absolutely incredible and stunning!'
Local Model Prediction: positive (Confidence: 92.42%)
Status: Local prediction accepted.
Text: 'It wasn't the worst thing ever, but I wouldn't watch it again.'
Local Model Prediction: negative (Confidence: 98.31%)
Status: Local prediction accepted.
